In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import scipy
import time
import gget
import anndata as an
import scanpy as sc
import scanpy.external as sce
import h5py
from collections import Counter
from Bio import Align
from Bio import SeqIO
from Bio.Align import PairwiseAligner
import pysam
from difflib import SequenceMatcher
import scipy.sparse as sp
import scipy.stats as stats
from matplotlib.ticker import PercentFormatter

sc.settings.verbosity = 2
# sc.logging.print_header()

In [3]:
fpath = "/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/pipeline_outputs/hyb_epi2me_final/hybrid/hybrid.read_summary.tsv"

df = pd.read_csv(fpath, sep='\t')
print(df.shape)
df.head()

(154491536, 12)


,read_id,uncorrected_barcode,corrected_barcode,quality_barcode,uncorrected_umi,corrected_umi,quality_umi,gene,transcript,start,end,chr
0,cc54fe5c-2dd3-4403-b361-6ab67656ccb9_0,GTGAGTAGCTCCAGGC,GTGAGTAGTTCCAGGC,"440/..($###$&(*,",GAATGATTACAG,GAATGATTACAG,"+++,00000000",-,-,38152,38457,GL000213.1
1,b81006c2-b975-4edb-a194-cfad8c189430_0,TCCTCAACAAGCGCGT,TCCTCAACAAGCGCGT,C?D@A=<>::995334,GGCCCTTTTATT,GGCCCTTTTATT,9723221=@>?A,-,-,4620,4933,GL000213.1
2,b81006c2-b975-4edb-a194-cfad8c189430_1,TCCTCAACAAGCGCGC,TCCTCAACAAGCGCGT,<@>;9:;688;;:8:(,TGCCCTTTTATT,GGCCCTTTTATT,(4BJSHKJAJPS,-,-,4659,4943,GL000213.1
3,2a8de785-8cca-48e3-a72b-13c4520494df_0,GTGAGTAGTTCCAGGC,GTGAGTAGTTCCAGGC,99944446768:5111,GCCACGGGCTGC,GCCACGGGCTGC,33///079888:,-,-,38149,38457,GL000213.1
4,b37758e5-66bc-4a78-9294-a0170695e257_0,CAAGTCGCATGACACG,CAAGTCGCATGACACG,0BB???DFIEEGFDEF,GTGGCCTCGGGG,GTGGCCTCGGGG,IKHLA?>@O?<5,ENSG00000271254,-,7025,8731,KI270711.1


In [3]:
epi2me_filtered_file = "/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/data/exo_vs_endo/reads/epi2me.myod.filtered.csv"

df = pd.read_csv(epi2me_filtered_file)
print(df.shape)
df.head()

(211761, 12)


,read_id,uncorrected_barcode,corrected_barcode,quality_barcode,uncorrected_umi,corrected_umi,quality_umi,gene,transcript,start,end,chr
0,24ae5696-3963-442a-8727-1cfcad58da48_0,TACTTCATCTCACTAC,TACTTCATCTCACTAC,LMHCDCIHLC?=>>AA,GAGTCATTCTGA,GAGTCATTCTGA,AEBFBCEDDFES,MYOD1,ENST00000250003,17720325,17721509,chr11
1,24c00180-b005-4640-9ccb-703a8ed006e8_0,CGGTTTGGTCAGCTCA,CGGTTTGGTCAGCTCA,CFSHKKJMGFPMEHII,GTGCTTGCTTGA,GTGCTTGCTTGA,GNJGEGFCIECS,-,ENST00000250003,17719782,17720313,chr11
2,eb029acf-0d34-421b-9bbd-d4ea6ad424b4_0,AGCCAGCCAATGTCAA,AGCCAGCCAATGTCAA,LGGHFSA=>333:>D?,CCCGTTGGGAAG,CCCGTTGGGAAG,::6773667?A@,-,ENST00000250003,17720298,17721509,chr11
3,4cf0d91e-afff-47db-902c-d7ffcd80b428_0,ACAGTGAGTCTCACGA,ACAGTGAGTCTCACGA,A;:::<&&9'((434:,CTAGGTACGTGT,CTAGGTACGTGT,;;HIEFDDDGSS,MYOD1,ENST00000250003,17719782,17720304,chr11
4,818257ad-3228-4c34-b31b-5e4ce6cb1925_0,GATGGGCACAGCGACT,GGATGGGCACAGCGAC,",8:@??@BBC@IMSHE",AAACATTGTACT,AAACATTGTACT,F=SCGA@=::==,MYOD1,ENST00000250003,17720900,17721509,chr11


In [15]:
df['read_id'].nunique()

211761

In [6]:
df['corrected_barcode'].nunique()

7453

In [14]:
tmp = df[df['corrected_barcode'] == 'ACAGTGAGTCTCACGA'].copy()

print(tmp['corrected_umi'].nunique())

tmp = tmp.sort_values(by='corrected_umi')

print(tmp.shape)
tmp.head()

25
(27, 12)


,read_id,uncorrected_barcode,corrected_barcode,quality_barcode,uncorrected_umi,corrected_umi,quality_umi,gene,transcript,start,end,chr
66110,28ef8767-9c00-4a89-9a8c-bf880137b081_0,ACAGTGAGTCTCACGA,ACAGTGAGTCTCACGA,GHKHED33<333::;=,AATGGTATTCAT,AATGGTATTCAT,EB@?AGOESCLQ,-,ENST00000250003,17719782,17720189,chr11
97654,c8a4e92c-96b8-4eb4-aab3-d0298799c707_0,ACAGTGAGTCTCACGA,ACAGTGAGTCTCACGA,IHGOHINHGKJ?<<@<,ACCATGTCGGAT,ACCATGTCGGAT,<<<OGCC<@EDI,-,ENST00000250003,17720923,17721509,chr11
97658,673ca8bf-8e8f-4e8d-bcca-bb81e8660add_0,ACAGTGAGTCTCACGA,ACAGTGAGTCTCACGA,DHGSLGISMLJPILDE,ACCATGTCGGAT,ACCATGTCGGAT,GFHFDEI9+5)3,-,ENST00000250003,17720923,17721509,chr11
188223,ac33d586-66bf-4212-be02-aed46a1edb2e_0,ACAGTGAGTCTCACGA,ACAGTGAGTCTCACGA,9=BAEBBBALFDFK::,ATCTTCAGCATG,ATCTTCAGCATG,::EE;66JKISE,-,ENST00000250003,17719847,17720296,chr11
8136,5b14d9c3-4c8a-4ad7-a5ad-872ceb399abb_0,ACAGTGAGTCTCACGA,ACAGTGAGTCTCACGA,GC@A8823941(((+(,CCCACGGCACAC,CCCACGGCACAC,(((01:<FA??H,MYOD1,ENST00000250003,17720900,17721509,chr11


In [5]:
barcode_gene_counts = '/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/data/exo_vs_endo/runs/minimap_transcripts/mmyod_barcode_gene_counts.tsv'

bdf = pd.read_csv(barcode_gene_counts, sep='\t')
print(bdf.shape)
bdf.head()

(28634, 4)


,barcode,gene,class,count
0,TGGTTTGAGAAGGTGT,ENST00000250003.4,primary,16
1,TGGTTTGAGAAGGTGT,MMYOD_ER_TX1,primary,20
2,AGGTTGGAGCCAGACT,ENST00000250003.4,primary,20
3,AGGTTGGAGCCAGACT,MMYOD_ER_TX1,primary,23
4,AATATGGAGGTCGATC,ENST00000250003.4,primary,16


In [7]:
bdf['barcode'].nunique()

7452